# DAC — Delaware peek

Scope: Delaware Medicare clinicians (adds `org_pac_id`, credential, specialty over NPPES).
Source: `data/sources/provider_registry/dac/DAC_NationalDownloadableFile.csv` (~800 MB, plain CSV).
Reading: chunked pandas, filtered to `State` = DE.

In [12]:
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
PATH = ROOT / "data/sources/provider_registry/dac/dac_DE.csv"
STATE_COL, STATE = "state", "DE"
sample = pd.read_csv(PATH, nrows=5, dtype=str)
print(sample.shape[1], "columns")
sample.head()

31 columns


,npi,ind_pac_id,ind_enrl_id,provider_last_name,provider_first_name,provider_middle_name,suff,gndr,cred,med_sch,...,adr_ln_1,adr_ln_2,ln_2_sprs,citytown,state,zip_code,telephone_number,ind_assgn,grp_assgn,adrs_id
0,1851302459,8426233636,I20110504000270,FONG,GLORIA,NaN,NaN,F,MD,OTHER,...,NaN,NaN,NaN,NEWARK,DE,19702,9122449474,Y,Y,DE19702XXXXNEXXXXXXXXXX00
1,1003255357,8921233875,I20250320000443,ZEYLIKMAN,YURIY,NaN,NaN,M,MD,OTHER,...,NaN,NaN,NaN,CLAYMONT,DE,19703,9724490540,Y,Y,DE19703XXXXCLXXXXXXXXXX00
2,1023351285,2567764202,I20250618001560,POLLOCK,BENJAMIN,NaN,NaN,M,DO,PHILADELPHIA COLLEGE OF OSTEOPATHIC MEDICINE,...,NaN,NaN,NaN,CLAYMONT,DE,19703,9724490540,Y,Y,DE19703XXXXCLXXXXXXXXXX00
3,1033386024,3072777986,I20250710002709,LE,CATHERINE,NaN,NaN,F,MD,OTHER,...,NaN,NaN,NaN,CLAYMONT,DE,19703,9724490540,Y,Y,DE19703XXXXCLXXXXXXXXXX00
4,1063698967,6709948730,I20250314001410,KATHIRESAN,KARTHIKEYANI,NaN,NaN,F,MD,OTHER,...,NaN,NaN,NaN,CLAYMONT,DE,19703,9724490540,Y,Y,DE19703XXXXCLXXXXXXXXXX00


In [13]:
sample.keys()

Index(['npi', 'ind_pac_id', 'ind_enrl_id', 'provider_last_name',
       'provider_first_name', 'provider_middle_name', 'suff', 'gndr', 'cred',
       'med_sch', 'grd_yr', 'pri_spec', 'sec_spec_1', 'sec_spec_2',
       'sec_spec_3', 'sec_spec_4', 'sec_spec_all', 'telehlth', 'facility_name',
       'org_pac_id', 'num_org_mem', 'adr_ln_1', 'adr_ln_2', 'ln_2_sprs',
       'citytown', 'state', 'zip_code', 'telephone_number', 'ind_assgn',
       'grp_assgn', 'adrs_id'],
      dtype='str')

In [14]:
# Scope to Delaware. Chunked scan (~800 MB).
parts = []
for chunk in pd.read_csv(PATH, dtype=str, chunksize=200_000):
    parts.append(chunk[chunk[STATE_COL] == STATE])
de = pd.concat(parts, ignore_index=True)
de.shape

(11894, 31)

In [15]:
# Fan-out, group linkage, specialty mix.
print("rows:", len(de), "| unique NPIs:", de["npi"].nunique())
print("rows per NPI (fan-out):", round(len(de) / de["npi"].nunique(), 2))
print("org_pac_id non-null %:", round(de["org_pac_id"].notna().mean() * 100, 1))
de["pri_spec"].value_counts().head(20)

rows: 11894 | unique NPIs: 7192
rows per NPI (fan-out): 1.65
org_pac_id non-null %: 90.4


pri_spec
NURSE PRACTITIONER                               1822
PHYSICIAN ASSISTANT                              1659
INTERNAL MEDICINE                                 741
PHYSICAL THERAPIST IN PRIVATE PRACTICE            722
CERTIFIED REGISTERED NURSE ANESTHETIST (CRNA)     650
DIAGNOSTIC RADIOLOGY                              593
FAMILY PRACTICE                                   571
CARDIOVASCULAR DISEASE (CARDIOLOGY)               471
CLINICAL SOCIAL WORKER                            333
EMERGENCY MEDICINE                                314
ANESTHESIOLOGY                                    275
MENTAL HEALTH COUNSELOR                           271
NEUROLOGY                                         249
CHIROPRACTIC                                      208
PSYCHIATRY                                        187
OPTOMETRY                                         178
CLINICAL PSYCHOLOGIST                             169
OBSTETRICS/GYNECOLOGY                             167
GENERAL SURGERY    

In [17]:
de["sec_spec_1"].value_counts().head(20)

sec_spec_1
INTERNAL MEDICINE                      364
HOSPITALIST                            117
CRITICAL CARE (INTENSIVISTS)           108
PEDIATRIC MEDICINE                      83
EMERGENCY MEDICINE                      44
INTERVENTIONAL RADIOLOGY                39
CARDIAC ELECTROPHYSIOLOGY               37
GENERAL SURGERY                         36
ANESTHESIOLOGY                          34
GERIATRIC MEDICINE                      34
DIAGNOSTIC RADIOLOGY                    28
INTERVENTIONAL CARDIOLOGY               26
PULMONARY DISEASE                       25
CARDIOVASCULAR DISEASE (CARDIOLOGY)     24
FAMILY PRACTICE                         21
PAIN MANAGEMENT                         17
ADDICTION MEDICINE                      15
ORTHOPEDIC SURGERY                      14
SPORTS MEDICINE                         12
OSTEOPATHIC MANIPULATIVE MEDICINE       10
Name: count, dtype: int64

In [18]:
de["sec_spec_2"].value_counts().head(20)

sec_spec_2
INTERNAL MEDICINE                       71
PEDIATRIC MEDICINE                      33
INTERVENTIONAL CARDIOLOGY               21
PULMONARY DISEASE                        7
NEUROLOGY                                6
FAMILY PRACTICE                          6
SLEEP MEDICINE                           5
HOSPITALIST                              5
NUCLEAR MEDICINE                         5
PAIN MANAGEMENT                          4
SPORTS MEDICINE                          2
PLASTIC AND RECONSTRUCTIVE SURGERY       1
PHYSICAL MEDICINE AND REHABILITATION     1
INTERVENTIONAL PAIN MANAGEMENT           1
SURGICAL ONCOLOGY                        1
PSYCHIATRY                               1
Name: count, dtype: int64

## Notes
- Grain: one row per NPI x enrollment x group x practice location (fan-out > 1 expected).
- `org_pac_id` groups clinicians into a billing entity (Q4); blank when address not tied to a group.
- `pri_spec` is free-text; map to the primary-care scope (HISTORY 001) empirically.